# 🧪 Participant Lab Guide
## End-to-End Databricks Hands-On Workshop

> ### 👋 You are all data practitioners today 💪
> For the next 3 hours, forget your job title. Whether you're a Data Engineer, Data Scientist, BI Analyst, or work in the Data Warehouse team — **today you are all of them.** You'll load data, explore it with AI, collaborate live, build a dashboard, and create your own AI assistant.
>
> **It's not as scary as you think.** Every step below has a copy-paste block and a ✅ checkpoint. If you can copy, paste, and click **Run**, you can do this. When you hit a green checkmark, give the room a 👍.

---

### 📖 How to read this guide
Each module follows the **same rhythm**, so you always know where you are:

- 🎯 **Goal** — one sentence on what you'll achieve
- 🛠️ **Steps** — numbered clicks, nothing skipped
- 📋 **Copy me** — code blocks you paste (don't type it out!)
- ✅ **Checkpoint** — what you should see when it worked
- 💡 **Stuck?** — the one or two things that trip people up

> **One-time setup (do this first in every notebook / SQL editor):** run the two lines below **once** at the top. After that, every script just names the table directly — no long `catalog.schema.` prefixes to type.
>
> ```sql
> USE CATALOG <your_catalog>;   -- your catalog from setup, e.g. ali_bank
> USE SCHEMA retail_360;
> ```
>
> Now `SELECT * FROM customers` just works. (Replace `<your_catalog>` with your name's catalog, e.g. `ali_bank`.)

---

## 🧰 Module 0 — Setup (5 min, do this first)

🎯 **Goal:** Create your personal catalog, schema, volume, and tables.

🛠️ **Steps**
1. In the left sidebar, make sure you're in the cloned Git folder for this workshop.
2. Open **`00-setup`** (at the top level of the repo).
3. At the top of the notebook, find the **`your_name`** widget → type your name in lowercase (e.g. `ali`).
4. Click **Run all** (▶▶ at the top).
5. Wait ~1 minute for the big green **✅ SETUP COMPLETE** box.

✅ **Checkpoint:** You see `✅ SETUP COMPLETE` and your catalog name (e.g. `ali_bank`). In the left sidebar under **Catalog**, you can find `<your_catalog>` → `retail_360` → 4 tables.

💡 **Stuck?**
- No green box? Check you typed a name in the widget and clicked **Run all**, not just one cell.
- Want a clean restart? Set the `reset` widget to `yes` and Run all again.

---

## 📥 Module 1 — Catalog, Schema & CSV Upload (20 min)

🎯 **Goal:** See how governed data lives in Unity Catalog, and upload your first table by hand.

🛠️ **Steps**
1. In the left sidebar, click **Catalog** 🗂️.
2. Expand `<your_catalog>` → `retail_360`. Notice the 4 tables setup created for you (accounts, transactions, products, branches).
3. Click the **`transactions`** table. Explore the tabs: **Columns**, **Sample Data**, **Details**. Notice the description we added — this is your governance layer.
4. Now **upload `customers.csv` yourself.** First, download it from your volume:
   - Go to `<your_catalog>` → `retail_360` → **Volumes** → `raw_files`.
   - Click **`customers.csv`** → **⋮** (or the download icon) → **Download**. Save it to your laptop.
5. Click **+ (Add / Create)** at the top of the sidebar → **Add data** → **Create or modify table**.
6. **Drag `customers.csv`** into the upload box (or browse to it).
7. Set the destination: **Catalog** = `<your_catalog>`, **Schema** = `retail_360`, **Table name** = `customers`.
8. Preview looks good? Click **Create table**.

📋 **Copy me** — first, set your working context **once** (top of a new SQL editor or a `%sql` cell). After this, you just name tables directly:
```sql
USE CATALOG <your_catalog>;   -- e.g. ali_bank
USE SCHEMA retail_360;
```

Now confirm your upload and document the table. Copy this block once; run the statements top to bottom:
```sql
-- 1) Confirm your table loaded (~800 customers across segments)
SELECT segment, COUNT(*) AS customers, ROUND(AVG(age), 1) AS avg_age
FROM customers
GROUP BY segment
ORDER BY customers DESC;

-- 2) Add a table description (documentation in one line)
COMMENT ON TABLE customers IS
  'Retail banking customers: demographics, segment, income band, home branch.';

-- 3) Tag the region COLUMN as PII. This is the attribute a policy will match on.
ALTER TABLE customers ALTER COLUMN region SET TAGS ('pii' = 'true');
```

✅ **Checkpoint:** `customers` now appears as a 5th table under `retail_360`, the query returns ~800 customers across segments, and on the **Details / Columns** tab the `region` column shows the `pii` tag.

🙌 **Your Turn — Part A: mask a column by tag (5 min)**
You've tagged `region` as PII. Now create a **policy** that automatically masks *any* column carrying that tag — using the `mask_pii` function that setup pre-built for you. This is **ABAC**: govern by attribute (the tag), not column-by-column.

Copy this whole block once. Run the statements **one at a time, top to bottom** — the comments tell you what to watch for after each:
```sql
-- STEP 1: See the raw values first (note the real region + name before masking)
SELECT customer_id, name, region, segment
FROM customers
LIMIT 5;

-- STEP 2: Create your own column-mask policy — matches the `pii` tag, applies the pre-built mask_pii
CREATE OR REPLACE POLICY mask_pii_columns
ON SCHEMA retail_360
COMMENT 'Mask any column tagged pii using the mask_pii function'
COLUMN MASK mask_pii
TO `account users`
FOR TABLES
MATCH COLUMNS has_tag_value('pii', 'true') AS pii_col
ON COLUMN pii_col;

-- STEP 3: Re-run the SELECT from STEP 1. `region` now shows ***REDACTED***,
--         while `segment` (untagged) is untouched. One tag, one policy, done.
SELECT customer_id, name, region, segment
FROM customers
LIMIT 5;

-- STEP 4: The attribute-based payoff — tag a SECOND column, then re-run STEP 3.
--         `name` is now masked too, with NO change to your policy.
ALTER TABLE customers ALTER COLUMN name SET TAGS ('pii' = 'true');
```

> 💡 That's the power of ABAC: you wrote **one** policy against a *tag*, and every current and future column with that tag is governed automatically.

---

🙌 **Your Turn — Part B: row-level security on `branches` (7 min)**
Column masks hide *columns*. **Row filters** hide *rows*. Now make each region team see only **their own branches** — using the `filter_by_region` function setup pre-built for you and the 5 `team_*` groups.

That function returns TRUE when you're an **admin**, or when you belong to the group that matches a row's `region` (`Central → team_central`, `East Coast → team_east_coast`, and so on).

Copy this block once. Run the statements **one at a time, top to bottom**:
```sql
-- STEP 1: See all rows first (as catalog owner you can see every region)
SELECT branch_id, branch_name, state, region
FROM branches
ORDER BY region;

-- STEP 2: Tag the region column on branches (the attribute the row filter matches on)
ALTER TABLE branches ALTER COLUMN region SET TAGS ('region_filter' = 'true');

-- STEP 3: Create your own row-filter policy — passes the tagged region column
--         into the pre-built filter_by_region function
CREATE OR REPLACE POLICY rls_branches_by_region
ON SCHEMA retail_360
COMMENT 'Row filter: each region team sees only their own branches'
ROW FILTER filter_by_region
TO `account users`
FOR TABLES
MATCH COLUMNS has_tag_value('region_filter', 'true') AS region_col
USING COLUMNS (region_col);

-- STEP 4: Re-run STEP 1. As an admin/owner you STILL see all rows (the function
--         lets admins through) — so this is what a regional, non-admin user hits.
SELECT branch_id, branch_name, state, region
FROM branches
ORDER BY region;
```

> 👀 **Want to actually watch it filter?** Ask the facilitator to add you to a single region group (e.g. `team_northern`) as a **non-admin** — then re-run STEP 1 and you'll see only Northern branches. The facilitator will demo this live so everyone sees the effect.

> 💡 Masks + row filters together = **column-level and row-level governance**, both driven by tags, both written as one policy on the schema.

💡 **Stuck?**
- Can't find **Create or modify table**? Use the **+** button at the very top of the left sidebar → **Add data**.
- **"Table or view not found"?** You didn't set context — run `USE CATALOG <your_catalog>;` then `USE SCHEMA retail_360;` once at the top, then re-run.
- **`Unknown tag policy key 'pii'` when creating the policy?** The governed tag isn't set up. Tell your facilitator to run `00-setup-ADMIN` — it creates the `pii` / `region_filter` governed tags and grants your group `ASSIGN`. (ABAC policies only work on *governed* tags.)
- `filter_by_region` not found? Re-run `00-setup` — it creates it in your schema.
- Policy error about multiple filters? You may have created it twice with different names — drop the extra: `DROP POLICY <name> ON SCHEMA retail_360;`.

---

## 🤖 Module 2 — Exploring Data with the Assistant (20 min)

🎯 **Goal:** Meet your AI pair-programmer. In this module you'll use the Databricks Assistant to **generate, explain, debug, translate, visualize and document** code — so you never have to remember syntax again. Today, *you're the analyst and the Assistant is your coder.*

> 💬 **The big idea:** You bring the *questions*. The Assistant writes the *code*. If you can describe what you want in plain English, you can analyse data here — SQL background or not.

### 🎛️ Setup (2 min)

🛠️ **Steps**
1. Create a new notebook: **+ → Notebook**. Rename it `my-exploration`.
2. Attach it to **Serverless** compute (top-right dropdown).
3. Set your working context by running this first cell:

📋 **Copy me** — same context block as Module 1 (run once at the top):
```sql
USE CATALOG <your_catalog>;   -- e.g. ali_bank
USE SCHEMA retail_360;
```
> Prefer Python? `spark.sql("USE CATALOG <your_catalog>"); spark.sql("USE SCHEMA retail_360")` does the same thing.

4. Meet the Assistant two ways:
   - **Inline** — press `Cmd/Ctrl + I` inside any cell (best for "write/fix this cell").
   - **Pane** — click the ✨ sparkle icon on the far-right edge (best for chatting about your data).
5. Type **`/`** inside the Assistant to see its slash-commands: `/explain`, `/fix`, `/doc`, `/optimize`, `/findTables`, `/prettify`. You'll use several below.

> 📇 **Know your columns (this makes the Assistant accurate).** The more precise your table/column names, the better the code it writes. Keep this handy:
> | Table | Key columns |
> |-------|-------------|
> | `customers` | `customer_id`, `age`, `state`, `region`, `segment`, `income_band`, `home_branch_id`, `join_date` |
> | `transactions` | `txn_id`, `customer_id`, `txn_date`, `amount_myr`, `channel`, `category`, `merchant` |
> | `accounts` | `account_id`, `customer_id`, `product_id`, `balance_myr`, `status` |
> | `products` | `product_id`, `product_name`, `product_type`, `profit_rate_pct` |
> | `branches` | `branch_id`, `branch_name`, `state`, `region` |
>
> 💡 **Golden rule for prompts:** name the **table** and the **exact column**. Say *"sum `amount_myr` from `transactions` grouped by `category`"*, not *"total spend by type"*. The Assistant can also read your schema with `/findTables` if you're unsure.

---

### 💪 The 6 superpowers of the Assistant (12 min)

Work through these in order — each is a new cell. This is a **guided tour**; the fun challenges come right after.

**① Generate — turn a question into SQL.**
In an empty cell, press `Cmd/Ctrl + I` and type this prompt (don't write SQL yourself). Notice it names the exact table and columns:
```
Using the `transactions` table, write SQL that sums the `amount_myr` column
and counts rows, grouped by `category`, ordered by total amount_myr descending.
```
Accept it and run. 🎉 You just wrote SQL without writing SQL. *(If it guesses a wrong column, tell it: "use the column amount_myr" — and it fixes itself.)*

**② Explain — understand any code.**
Paste the query below into a new cell, highlight it, open the Assistant and type `/explain`:
```sql
%sql
SELECT c.segment,
       ROUND(SUM(t.amount_myr), 2) AS total_spend,
       COUNT(DISTINCT c.customer_id) AS customers,
       ROUND(SUM(t.amount_myr) / COUNT(DISTINCT c.customer_id), 2) AS spend_per_customer
FROM customers c
JOIN transactions t ON c.customer_id = t.customer_id
GROUP BY c.segment
ORDER BY total_spend DESC;
```
Read the plain-English explanation. Great for code someone *else* wrote.

**③ Debug — let the Assistant fix a broken query.**
Paste this **intentionally broken** SQL and run it. It will fail 💥 — that's the point:
```sql
%sql
SELECT segment, COUNT(*) AS cust
FROM customers
WHERE age > 30
GROUP BY segmnt
ORDER BY cust DES;
```
When the error appears, click **Diagnose error** (or open the Assistant and type `/fix`). Review the proposed diff, **Accept**, and re-run. It should spot the typo'd `segmnt` and the broken `DES`. *You just debugged code you didn't write.*

**④ Translate — SQL ↔ Python, same logic.**
Highlight your working query from ① and ask the Assistant:
```
Convert this SQL to PySpark DataFrame code using the existing `spark` session.
Read the table with spark.table("transactions") and keep the same columns.
```
Notice the logic is preserved, only the syntax changes. Run it to confirm you get the same numbers.

**⑤ Visualize — describe a chart in words.**
Run the copy-me below (it uses real columns `channel`, `amount_myr`), then use the result's **+ / Visualization** to make a bar chart — *or* ask the Assistant: *"suggest a visualization for this result; the columns are `channel`, `avg_txn` and `txns`."*
```sql
%sql
SELECT t.channel,
       ROUND(AVG(t.amount_myr), 2) AS avg_txn,
       COUNT(*) AS txns
FROM transactions t
GROUP BY t.channel
ORDER BY txns DESC;
```

**⑥ Document — auto-comment your work.**
Highlight any query and type `/doc` in the Assistant. It adds clear inline comments — instant documentation for the teammate who inherits your notebook.

✅ **Checkpoint:** You've used the Assistant to **generate**, **explain**, **fix a failing query**, **translate to PySpark**, **suggest a chart**, and **document** — the six things you'll do every day. If your cells ①–⑥ ran, give the room a 👍.

---

🙌 **Your Turn — Build a Pipeline with the Assistant (6 min)**

Now *you* drive — and you'll build a real **data pipeline**, prompt by prompt. No copy-paste SQL to solve it for you; the Assistant writes the SQL, **you** describe what you want.

> 🥉🥈🥇 **The medallion idea (30-sec version).** Real data teams refine data in layers:
> - **Bronze** = raw data as it landed. *(That's the 5 tables setup already loaded for you — treat them as bronze.)*
> - **Silver** = cleaned & combined — the useful, trustworthy tables.
> - **Gold** = business-ready — one table that answers real questions.
>
> Today **you build the silver layer**, then in Module 3 you'll team up to build **gold**. Plain SQL `CREATE TABLE` — no fancy pipeline tools.

> 👥 **Split the room.** Your facilitator will put you in **Group 🅰️** or **Group 🅱️**. Build **only your group's table** below — you'll combine them with a partner in Module 3.

**🧭 How to prompt the Assistant to build a table (read this once):**
- **Name the exact tables and columns** you want (use the column cheat-sheet above).
- **Ask for a table, not just a query** — say *"write a `CREATE OR REPLACE TABLE ... AS SELECT ...`"* so the result is saved, not just displayed.
- **Iterate — that's the fun part.** Run it, look at the result, then refine: *"now add a column for..."*, *"only keep rows where..."*, *"round that to 2 decimals."* The Assistant remembers the conversation. 🧠

---

**🅰️ Group A — build `silver_customer_dim` (one row per customer).**
This is your **customer dimension**: everything about a customer in one clean row — their details, their home branch, and a summary of what products they hold. Paste this as your **starter prompt** (press `Cmd/Ctrl + I` in an empty cell), run it, then refine:
```
Write SQL that creates a table called silver_customer_dim using
CREATE OR REPLACE TABLE. One row per customer. Start from the `customers`
table, LEFT JOIN `branches` on customers.home_branch_id = branches.branch_id
to add branch_name, state and region. Then LEFT JOIN an aggregate of the
`accounts` table grouped by customer_id that gives products_held =
COUNT(account_id) and total_balance_myr = ROUND(SUM(balance_myr), 2).
Keep customer_id, name, age, segment, income_band, branch_name, state,
region, products_held, total_balance_myr.
```
Then try a **refinement** of your own, e.g.: *"add a column `has_balance` that is true when total_balance_myr > 0"* or *"also add the customer's join_date."*

✅ **Checkpoint (Group A):** `SELECT COUNT(*) FROM silver_customer_dim;` returns **~800** (one per customer), and `SELECT * FROM silver_customer_dim LIMIT 5;` shows branch + product-holding columns filled in. `silver_customer_dim` now appears under `retail_360`.

---

**🅱️ Group B — build `silver_transactions` (one clean row per transaction).**
This is your **cleaned transactions fact** — the same transactions, but tidied and ready to analyse. Paste this as your **starter prompt**, run it, then refine:
```
Write SQL that creates a table called silver_transactions using
CREATE OR REPLACE TABLE, selecting from the `transactions` table.
Keep txn_id, customer_id, amount_myr, channel, category, merchant.
Cast txn_date to a DATE column called txn_date, and add a column
txn_month = date_trunc('month', txn_date). Only keep rows where
amount_myr > 0.
```
Then try a **refinement** of your own, e.g.: *"add a column `is_large` that is true when amount_myr > 1000"* or *"also add the day-of-week name of txn_date."*

✅ **Checkpoint (Group B):** `SELECT COUNT(*) FROM silver_transactions;` returns close to the raw transaction count (a bit fewer if any non-positive amounts were dropped), and `SELECT * FROM silver_transactions LIMIT 5;` shows a clean `txn_date` and the new `txn_month`. `silver_transactions` now appears under `retail_360`.

> 🎯 **Stretch (if you're flying):** highlight your finished `CREATE TABLE` and type `/doc` to auto-comment it, or `/explain` to have the Assistant walk a teammate through it. Done early? Peek at the *other* group's starter prompt so you understand both halves before Module 3.

💡 **Stuck?**
- Assistant not showing? Click the ✨ sparkle icon on the far right edge, or press `Cmd/Ctrl + I` in a cell.
- "Table not found"? Make sure you ran the `USE CATALOG` / `USE SCHEMA` cell first (Setup step 3), and don't prefix tables with a catalog — you're already *in* `retail_360`.
- Assistant's SQL not perfect? That's normal — **tell it what's wrong** ("that used the wrong column, use `amount_myr`" or "make it CREATE OR REPLACE TABLE, not a SELECT") and it revises. Conversation beats perfection.
- Table didn't save? Make sure the SQL actually starts with `CREATE OR REPLACE TABLE silver_...` — a plain `SELECT` only shows results, it doesn't create a table.

---

## 🤝 Module 3 — Collaborate to Build the Gold Layer (15 min)

🎯 **Goal:** Team up — one 🅰️ + one 🅱️ — and combine your two silver tables into a single **gold** table, live and together, like Google Docs for data.

> 👥 **Pair up: one Group A + one Group B.** You each built half the pipeline in Module 2. Now you'll join forces (literally 😄). Decide whose notebook you'll build **gold** in — call that person the **Driver**, the other the **Co-pilot**. You'll swap later.

🛠️ **Steps**

**① Share both ways (so you can both see everything):**
1. **Group A:** open your `my-exploration` notebook → **Share** (top-right) → enter your partner's email → **Can Edit** → **Add**.
2. **Group B:** do the same — share *your* notebook back to your partner with **Can Edit**.
3. Also share your **catalogs** so the join works: each person runs this once in their own notebook (replace `<partner_catalog>`), so your partner can read your silver table:
   ```sql
   GRANT USE CATALOG, USE SCHEMA, SELECT
   ON CATALOG `<your_catalog>` TO `<partner_email>`;
   ```
4. Open your partner's notebook (check **Recents** or **Shared with me**). You'll both see each other's **cursors move in real time**. 👀 Wave at each other with a cell!

**② Build `gold_customer_360` together (the Driver types, both watch):**
Your gold table = **A's customer dimension + B's transaction summary**, one clean row per customer — the business-ready table a dashboard or Genie would love.

> This join **crosses two catalogs** (A's silver + B's silver), so here you *do* write the full `catalog.schema.table` names. Paste this as your **starter prompt** (`Cmd/Ctrl + I`), swapping in each catalog:
```
Write SQL that creates a table gold_customer_360 in my catalog using
CREATE OR REPLACE TABLE. Take every column from
`<A_catalog>`.retail_360.silver_customer_dim as the base (call it d).
LEFT JOIN an aggregate of `<B_catalog>`.retail_360.silver_transactions
grouped by customer_id that gives: txn_count = COUNT(txn_id),
total_spend_myr = ROUND(SUM(amount_myr), 2),
avg_txn_myr = ROUND(AVG(amount_myr), 2),
last_txn_date = MAX(txn_date). Join on customer_id. One row per customer.
```
Run it, look at the result together, then **refine as a pair** — e.g. *"add a column `spend_per_product` = total_spend_myr / products_held"* or *"sort by total_spend_myr descending."*

✅ **Checkpoint:** `SELECT * FROM gold_customer_360 ORDER BY total_spend_myr DESC LIMIT 10;` shows your top spenders with **both** their profile (segment, branch, products_held) **and** their spend summary (txn_count, total_spend_myr) — in one row. You just built bronze → silver → **gold**, as a team, across two people's data. 🥇

🙌 **Your Turn — make it collaborative & fun (5 min)**
1. **💬 Co-write the headline.** Together, add a `%md` cell titled *"What our gold table reveals"* and write one sentence naming your **top-spending segment**. Both of you type in it at once — watch the words appear live.
2. **🗨️ Leave a comment.** Highlight the gold-table cell → **Comment** → `@mention` your partner with a question like *"should we add average balance too?"* Watch the comment pop up on their screen, and reply inline.
3. **🔁 Swap the Driver.** Now the **other** person drives: in *their* notebook, build a tiny `gold_top_customers` = `SELECT * FROM gold_customer_360 WHERE total_spend_myr > 5000` (ask the Assistant). Same data, roles reversed.
4. **🏆 Bragging rights.** As a pair, drop a comment with the single most surprising number you found. Facilitator will ask a couple of pairs to share.

💡 **Stuck?**
- Partner can't see the notebook? Double-check the email and that permission is **Can Edit**.
- `PERMISSION_DENIED` / can't read their silver table? The owner needs to run the `GRANT ... ON CATALOG` line in Step ①.3 (or ask the facilitator to run it).
- One silver table missing? Whoever's `silver_customer_dim` or `silver_transactions` didn't build — jump back to Module 2's starter prompt and create it, then re-run the gold prompt.
- Table names don't match? Confirm A built `silver_customer_dim` and B built `silver_transactions` (exact names) — the gold prompt references both.

---

## 🧠 Module 4 — AutoML + Inference *(watch the facilitator)* (15 min)

🎯 **Goal:** See how the platform goes from a table to a deployed ML model — **"Next Best Offer"** — in minutes.

> 🎬 **This module is a demo.** Sit back and watch the facilitator. No hands-on needed — just follow the story.

**What you'll see the facilitator do:**
1. Point **AutoML** at a `customer_360_features` table where the target column is **`next_best_offer`** (which product to recommend each customer next).
2. Start a **Classification** experiment — AutoML tries dozens of models automatically.
3. Review the **leaderboard** — best model on top, with a generated notebook for each.
4. **Register** the winning model to Unity Catalog.
5. Deploy it to a **real-time serving endpoint** and hit it with a REST call to get a live prediction.

💬 **Why it matters:** Data → model → production endpoint on **one platform**. No handoff to a separate ML team, no separate MLOps toolchain. The same governed tables you explored become the fuel for ML.

🙌 **Your Turn — think about it (discussion)**
No hands-on here — just get your brain going for the round-table later:
1. **Spot a feature.** Looking at the columns the facilitator used, name **one more feature** you'd add to improve "Next Best Offer" (e.g. months since last product opened).
2. **Name the action.** If the model says a customer's next best offer is *Home Financing-i*, **what should the bank actually do** with that prediction?

✅ **Checkpoint (mental):** You understand that "Next Best Offer" turns your customer + transaction data into a prediction the business can act on — and that any analyst here could kick off AutoML.

---

## ☕ Break (5 min)

Stretch, grab a drink. When you're back, we build dashboards. Everyone with a ✅ so far, thumbs up!

---

## 📊 Module 5 — Building a Dashboard (25 min)

🎯 **Goal:** Turn the **gold table you built in Module 3** into a shareable AI/BI dashboard for stakeholders.

> 🥇 **You're building on your gold layer.** Your `gold_customer_360` table — one business-ready row per customer, profile + spend summary — is the star of this dashboard. You'll also add the raw **transactions** table so you can still chart spend by category, channel and month. Gold answers "who are my customers?"; transactions answer "what did they do?".
>
> 👉 **Which catalog?** Use whichever catalog your pair built `gold_customer_360` in during Module 3 (it may be your partner's — you were granted `SELECT` on it). Replace `<gold_catalog>` below with that catalog. *(No gold table? Build it fast with Module 3's starter prompt, or fall back to your own `customers` + `transactions`.)*

🛠️ **Steps**
1. Left sidebar → **Dashboards** → **Create dashboard**. Name it `<your_name> Retail 360`.
2. Go to the **Data** tab → **+ Add data** → add **`gold_customer_360`** (your gold table) and **transactions** from `<gold_catalog>.retail_360`.
3. Go to the **Canvas** tab. Click **Add a visualization** (chart icon).
4. Build three charts by describing them or dragging fields:

📋 **Copy me** — if you'd rather define a dataset with SQL, use the **Data** tab → **Create from SQL**. (Dashboards don't share a notebook's `USE` context, so here we name the catalog in full — replace `<gold_catalog>`.) This joins your **gold** customer table to the raw transactions so every chart below works:
```sql
SELECT g.customer_id,
       g.segment,
       g.state,
       g.region,
       g.products_held,
       g.total_balance_myr,
       g.total_spend_myr,
       t.category,
       t.channel,
       t.amount_myr,
       t.txn_date
FROM `<gold_catalog>`.retail_360.gold_customer_360 g
JOIN `<gold_catalog>`.retail_360.transactions t
  ON g.customer_id = t.customer_id;
```

5. **Chart 1 — Bar:** Total `amount_myr` by `category`.
6. **Chart 2 — Map or Bar:** Customers (or spend) by `state`.
7. **Chart 3 — Line:** `amount_myr` over `txn_date` (by month).
8. Add a **Filter** widget on `segment` (so viewers can slice by customer segment).
9. Click **Publish** (top-right) → then **Share** → add your neighbour as a viewer.

✅ **Checkpoint:** A published dashboard with 3 charts + a working segment filter, all sourced from your **gold** table. Toggle the filter and watch every chart update. Your neighbour can open your link.

🏆 **Your Turn — Dashboard Design-Off! (10 min)**

Now make it *shine*. You have **10 minutes** to build the **nicest dashboard in the room** off your gold table. When time's up, neighbours vote (or the facilitator judges) against the scorecard below. 🥇🥈🥉

**📋 The scorecard (100 pts):**

| # | Criterion | Points | How to nail it |
|---|-----------|:------:|----------------|
| 1 | 🎨 **Bank Rakyat brand colours** | 25 | Style the dashboard in Bank Rakyat's brand colours (blues & oranges), not the default palette. Set a **colour theme** and use it consistently across charts. |
| 2 | 📑 **Multi-page** | 20 | Split into **≥2 pages/tabs** — e.g. an *Overview* page (KPIs + spend) and a *Customers* page (segments, states). |
| 3 | 🎚️ **Global filters** | 20 | Add **≥2 filter widgets** (e.g. `segment` **and** `state`/`region`) that cross-filter **every** chart on the page. |
| 4 | 🔢 **KPI counter row** | 15 | A row of **big-number tiles** across the top: total customers, total spend (MYR), avg balance, products held. |
| 5 | 📊 **Chart variety** | 20 | Use **≥4 different chart types**, and include a **map of Malaysian states**. |

**⏱️ How to spend your 10 minutes:**
- **0–2 min:** add a KPI counter row (total customers, total spend, avg `total_balance_myr`).
- **2–5 min:** set the **brand colour theme** and restyle your charts (blues & oranges).
- **5–8 min:** add a **second page** and a **state map**; move the customer charts there.
- **8–10 min:** wire up **two global filters** (segment + state) and check every chart reacts.

> 💡 **Pro moves for style points:** give the dashboard a title + section headers, add a `%md`/text widget with your headline insight, and try the dashboard **Assistant** ("*chart of average transaction amount by channel, styled in orange*") to add a chart hands-free.

💡 **Stuck?**
- No data in a chart? Confirm you added `gold_customer_360` + `transactions` in the **Data** tab (or created the SQL dataset) first.
- Chart looks empty? Check the field you dropped on the axis matches the dataset (e.g. `amount_myr` and `total_balance_myr` are numeric).
- Can't find the colour theme? It's in the dashboard's **⋮ / settings** (or the paint-roller/theme icon on the canvas toolbar) → pick custom colours.
- No `gold_customer_360`? Use the catalog your pair built it in (Module 3), or rebuild it in seconds with Module 3's starter prompt.

---

## 💬 Module 6 — Building Your Own Genie Space (25 min)

🎯 **Goal:** Package the **gold table you built in Module 3** + business context so anyone can ask it questions in plain English.

> 🥇 **Genie loves a gold table.** `gold_customer_360` is already business-ready — one clean row per customer with spend and balance pre-summarised — so Genie gives faster, more reliable answers than wrangling raw tables. You'll add it as the star, plus `transactions` and `products` for the detail questions.
>
> 👉 **Which catalog?** Use whichever catalog your pair built `gold_customer_360` in during Module 3 (it may be your partner's — you were granted `SELECT` on it). Replace `<gold_catalog>` below with that catalog.

🛠️ **Steps**
1. Left sidebar → **Genie** → **New** (or **+ Genie space**).
2. Name it `<your_name> Retail Genie`.
3. **Add tables:** from `<gold_catalog>.retail_360` → add **gold_customer_360** (the star), **transactions** and **products**.
4. **Add instructions** (this is what makes Genie smart). Paste these:

📋 **Copy me** — general instructions:
```
This Genie space answers questions about a retail bank's customers, their
spend, balances and products. The main table is gold_customer_360 — one
business-ready row per customer.
- Prefer gold_customer_360 for customer-level questions. It already has
  per-customer totals: total_spend_myr, avg_txn_myr, txn_count,
  total_balance_myr, products_held, plus segment, state, region.
- "spend" or "spending" at the customer level means total_spend_myr from
  gold_customer_360; for spend by category or channel, use the
  transactions table (amount_myr) joined on customer_id.
- income_band values are: <3K, 3K-5K, 5K-8K, 8K-12K, 12K-20K, >20K.
- Always show amounts in MYR (RM) and round to 2 decimals.
- A customer's "segment" is one of: Mass, Mass Affluent, Affluent, Youth, Senior.
```

5. **Add sample questions** (SQL example queries teach Genie your patterns). Add these as sample/trusted questions:

📋 **Copy me** — sample questions to seed:
```
1. Which customer segment has the highest average total_spend_myr in gold_customer_360?
2. What is the total transaction amount by product category?
3. Which states have the most customers?
4. List the top 10 customers by total_spend_myr.
5. What is the average total_balance_myr by segment?
```

6. Click into the chat and **test it** — ask: *"Which segment spends the most on Dining?"* and *"How many customers are in Selangor?"*
7. Refine one instruction if an answer looks off, then re-ask.
8. **Share** the Genie space with your neighbour.

✅ **Checkpoint:** Genie answers at least two of your questions correctly with a table/chart — using your **gold** table — and your neighbour can open the space.

🙌 **Your Turn — try these (5 min)**
1. **Teach Genie a new term.** Add one instruction, e.g. *"A 'digital customer' is one whose transactions are mostly on Mobile App or Internet Banking."* Then ask *"How many digital customers do we have?"*
2. **Add your own trusted question.** Create one sample question that matters to you (e.g. *"Average balance by product type"*) and save it.
3. **Push it harder.** Ask a comparison: *"Compare total spend between Youth and Senior segments."* Refine an instruction if the answer looks off.

💡 **Stuck?**
- Wrong answer? Add or sharpen an **instruction** (e.g. define the term it got wrong), then ask again.
- Genie can't find a column? Make sure the relevant table is added to the space.

---

## 🦾 Module 7 — Knowledge Assistant + Supervisor Agent (30 min)

🎯 **Goal:** Build a no-code AI assistant — one that answers from your **documents** (RAG), and one that **orchestrates across your Genie spaces**.

### Part A — Knowledge Assistant (RAG on product PDFs) — 15 min

🛠️ **Steps**
1. Left sidebar → **Agents** (Agent Bricks) → **Knowledge Assistant** → **Create**.
2. Name it `<your_name> Product Helper`.
3. **Add a knowledge source** → point it at your product docs in the volume:
   - Path: `/Volumes/<your_catalog>/retail_360/raw_files/product_docs/`
   - This folder holds the bank's **Product Disclosure Sheet** PDFs.
4. Give it a short description: *"Answers customer questions about our financing and deposit products using the official product disclosure sheets."*
5. Let it build (it chunks + indexes the PDFs automatically). Then **test in the chat**:

📋 **Copy me** — questions to ask your Knowledge Assistant:
```
1. What is the profit rate for Personal Financing-i?
2. What is the minimum deposit for the Term Deposit account?
3. What are the eligibility requirements for Vehicle Financing-i?
4. Which Shariah concept does Home Financing-i use?
```

✅ **Checkpoint (Part A):** The assistant answers a product question and **cites the PDF** it pulled the answer from.

🙌 **Your Turn — try these (5 min)**
1. **Cross-document question.** Ask: *"Compare the profit rate of Personal Financing-i and Home Financing-i."* — it should pull from **two** PDFs.
2. **Dig into fees.** Ask about the **fees or late payment charges** for any one product and check the citation.

### Part B — Supervisor Agent (orchestrates your Genies) — 15 min

🛠️ **Steps**
6. Left sidebar → **Agents** → **Multi-Agent Supervisor** → **Create**. Name it `<your_name> Bank Assistant`.
7. **Add agents** for it to orchestrate:
   - Your **`<your_name> Retail Genie`** (from Module 6) — for structured data questions.
   - Your **`<your_name> Product Helper`** (from Part A) — for product document questions.
8. Give the supervisor a description: *"Routes banking questions to the right specialist: data questions to the Retail Genie, product questions to the Product Helper."*
9. **Test the routing** in chat — ask a mix and watch which specialist it calls:

📋 **Copy me** — questions that test orchestration:
```
1. How many customers do we have in Johor?          (→ Retail Genie)
2. What is the profit rate on Personal Financing-i?  (→ Product Helper)
3. Which segment spends most, and what financing product could we offer them?
                                                     (→ uses both!)
```

✅ **Checkpoint (Part B):** The Supervisor routes each question to the right agent, and the combined question touches both. You just built a working assistant in under 30 minutes — grounded on your data and documents, governed by Unity Catalog.

🙌 **Your Turn — try these (5 min)**
1. **Force a hand-off.** Ask a data question then a product question back-to-back, and watch which specialist each one routes to.
2. **One question, both agents.** Ask: *"Which segment holds the fewest financing products, and what are the eligibility requirements for Personal Financing-i?"*

💡 **Stuck?**
- Knowledge Assistant returns nothing? Confirm the volume path is exact and the PDFs are in `product_docs/` (setup put them there).
- Supervisor doesn't route well? Improve each sub-agent's **description** — the supervisor uses descriptions to decide who answers.

---

## 🏁 Wrap-up & Next Steps (10 min)

🎉 **Look what you built this morning — as one person, in one browser tab:**

| Module | You built |
|--------|-----------|
| 1 | A governed catalog + uploaded your own table |
| 2 | AI-assisted data exploration |
| 3 | Live collaboration on shared data |
| 4 | (Saw) an ML model for Next Best Offer |
| 5 | A published, shareable dashboard |
| 6 | Your own natural-language Genie space |
| 7 | A RAG knowledge assistant + a supervisor agent |

**Round-table:** think of **one use case** from your real work at the bank you'd bring back to this platform. We'll go around the room.

> One platform, one morning — you built what usually takes multiple teams and weeks. **What's your first project?**

---

### 📎 Appendix — Quick reference

**Your objects**
- Catalog: `<your_catalog>` (e.g. `ali_bank`)
- Schema: `retail_360`
- Volume: `/Volumes/<your_catalog>/retail_360/raw_files/`
- Product docs: `/Volumes/<your_catalog>/retail_360/raw_files/product_docs/`

**Tables**
| Table | What's in it |
|-------|--------------|
| `customers` | demographics, segment, income_band, home branch (you upload in M1) |
| `accounts` | product holdings per customer + balances |
| `transactions` | ~15k transactions: amount, channel, category, merchant |
| `products` | product catalog with profit rates |
| `branches` | branch + state + region |
